In [1]:
import openmeteo_requests
import pandas as pd
import numpy as np
from retry_requests import retry

# 1. Setup the API client with retry logic
# If the connection drops, it will try 5 times before giving up
retry_session = retry(backoff_factor=0.2, retries=5)
openmeteo = openmeteo_requests.Client(session=retry_session)

# 2. Define location and variables (Lisbon, Portugal)
# We are fetching 10 years of hourly data (approx. 87,600 rows)
params = {
    "latitude": 38.7167,
    "longitude": -9.1333,
    "start_date": "1990-01-01",
    "end_date": "2024-01-01",
    "hourly": ["temperature_2m", "relative_humidity_2m", "surface_pressure", "precipitation", "wind_speed_10m"],
    "timezone": "UTC"
}   

print("Fetching data from Open-Meteo API...")
responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)
response = responses[0]

# 3. Process the hourly data
hourly = response.Hourly()

# THE FIX: 'inclusive="left"' is the modern Pandas 2.0+ replacement for 'closed="left"'
date_range = pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit="s"),
    end=pd.to_datetime(hourly.TimeEnd(), unit="s"),
    freq=pd.Timedelta(seconds=hourly.Interval()),
    inclusive="left"
)

# Organize raw arrays into a dictionary
weather_dict = {
    "date": date_range,
    "temperature": hourly.Variables(0).ValuesAsNumpy(),
    "humidity": hourly.Variables(1).ValuesAsNumpy(),
    "pressure": hourly.Variables(2).ValuesAsNumpy(),
    "precipitation": hourly.Variables(3).ValuesAsNumpy(),
    "wind_speed": hourly.Variables(4).ValuesAsNumpy()
}

# 4. Create the DataFrame
df_hourly = pd.DataFrame(data=weather_dict)

# 5. Data Validation & Export
print(f"Successfully collected {len(df_hourly)} hours of data.")
print("-" * 30)
print(df_hourly.head()) # Shows the first 5 rows
print("-" * 30)

# Save the raw data to your 'data' folder
# This ensures we don't have to call the API every time we restart
df_hourly.to_csv("../data/weather_raw.csv", index=False)
print("Data saved to data/weather_raw.csv")

Fetching data from Open-Meteo API...
Successfully collected 298056 hours of data.
------------------------------
                 date  temperature   humidity     pressure  precipitation  \
0 1990-01-01 00:00:00    13.259500  89.145416  1013.384705            0.0   
1 1990-01-01 01:00:00    13.259500  88.265274  1013.384705            0.3   
2 1990-01-01 02:00:00    13.209499  88.847046  1013.085083            0.6   
3 1990-01-01 03:00:00    13.109500  88.838829  1013.182617            0.7   
4 1990-01-01 04:00:00    13.009500  88.536697  1012.683533            0.7   

   wind_speed  
0   12.181624  
1   12.979984  
2   13.044722  
3   15.273505  
4   13.324863  
------------------------------
Data saved to data/weather_raw.csv
